# Insurance Agent Payroll & Commission Calculator

**Objective:** To build a rule-based Python calculator that processes monthly sales data for insurance agents and generates a payslip. 

**Business Rules:**
* Agents receive a base monthly salary.
* Commission is volume-based and varies by product type (Life, Health, Auto).
* Net pay is calculated using 2026/2027 Scottish Income Tax bands.

In [1]:
# --- SYSTEM CONFIGURATION ---

# 2026/2027 Scottish Income Tax Bands (Yearly)
# For simplicity in this project, we will divide the yearly thresholds by 12 for monthly calculations
tax_brackets_yearly = {
    "personal_allowance": 12570,
    "starter_rate": {"rate": 0.19, "threshold_max": 16537},      
    "basic_rate": {"rate": 0.20, "threshold_max": 29526},        
    "intermediate_rate": {"rate": 0.21, "threshold_max": 43662}, 
}

# Commission payouts per policy sold (£)
commission_rates = {
    "life_insurance": 150.00,
    "health_insurance": 50.00,
    "home_auto_insurance": 20.00
}

print("System Configuration Loaded Successfully.")

System Configuration Loaded Successfully.


In [2]:
# --- MONTHLY INPUT DATA ---

agent_data = {
    "agent_id": "AG-001",
    "name": "Jane Doe",
    "base_salary_monthly": 2000.00,
    "sales_this_month": {
        "life_insurance": 4,          # High value, low volume
        "health_insurance": 12,       # Medium value
        "home_auto_insurance": 25     # Low value, high volume
    }
}

print(f"Loaded data for Agent: {agent_data['name']}")

Loaded data for Agent: Jane Doe


In [3]:
# --- FUNCTION: CALCULATE COMMISSION ---

def calculate_commission(sales_data, rates):
    """
    Calculates total commission based on sales volume and product payout rates.
    """
    total_commission = 0.0
    
    print("--- Commission Breakdown ---")
    
    # Loop through each product the agent sold
    for product, quantity in sales_data.items():
        # Look up the payout rate for this specific product
        # We use .get() so if a product isn't found, it safely defaults to £0
        rate = rates.get(product, 0)
        
        # Calculate commission for this specific product line
        product_commission = quantity * rate
        
        # Add it to our running total
        total_commission += product_commission
        
        # Print a formatted string to show the math
        formatted_name = product.replace('_', ' ').title()
        print(f"{formatted_name}: {quantity} sold at £{rate:.2f} = £{product_commission:.2f}")
        
    return total_commission

# --- TEST THE FUNCTION ---

# We pass the specific data dictionaries we created in previous cells into our new function
monthly_commission = calculate_commission(agent_data["sales_this_month"], commission_rates)

print("-" * 28)
print(f"Total Commission Earned: £{monthly_commission:.2f}")

--- Commission Breakdown ---
Life Insurance: 4 sold at £150.00 = £600.00
Health Insurance: 12 sold at £50.00 = £600.00
Home Auto Insurance: 25 sold at £20.00 = £500.00
----------------------------
Total Commission Earned: £1700.00


In [4]:
# --- FUNCTION: CALCULATE DEDUCTIONS & NET PAY ---

def calculate_net_pay(monthly_gross, tax_brackets):
    """
    Calculates monthly tax and National Insurance (NI) based on annualized salary,
    using progressive tax bands.
    """
    # 1. Annualize the salary for accurate bracket calculation
    yearly_gross = monthly_gross * 12
    yearly_tax = 0.0
    
    # Extract the personal allowance (tax-free amount)
    allowance = tax_brackets["personal_allowance"]
    
    # 2. Calculate Progressive Income Tax
    if yearly_gross > allowance:
        taxable_income = yearly_gross - allowance
        
        # Starter Rate (19%)
        starter_max = tax_brackets["starter_rate"]["threshold_max"] - allowance
        if taxable_income > 0:
            taxed_amount = min(taxable_income, starter_max)
            yearly_tax += taxed_amount * tax_brackets["starter_rate"]["rate"]
            taxable_income -= taxed_amount
            
        # Basic Rate (20%)
        basic_max = tax_brackets["basic_rate"]["threshold_max"] - tax_brackets["starter_rate"]["threshold_max"]
        if taxable_income > 0:
            taxed_amount = min(taxable_income, basic_max)
            yearly_tax += taxed_amount * tax_brackets["basic_rate"]["rate"]
            taxable_income -= taxed_amount
            
        # Intermediate Rate (21%)
        intermediate_max = tax_brackets["intermediate_rate"]["threshold_max"] - tax_brackets["basic_rate"]["threshold_max"]
        if taxable_income > 0:
            taxed_amount = min(taxable_income, intermediate_max)
            yearly_tax += taxed_amount * tax_brackets["intermediate_rate"]["rate"]
            taxable_income -= taxed_amount
            
        # (For simplicity in this portfolio piece, we stop at the intermediate band)

    # 3. Calculate National Insurance (simplified 8% above allowance for 2026/2027)
    yearly_ni = 0.0
    if yearly_gross > allowance:
        yearly_ni = (yearly_gross - allowance) * 0.08

    # 4. Convert back to monthly figures
    monthly_tax = yearly_tax / 12
    monthly_ni = yearly_ni / 12
    net_pay = monthly_gross - monthly_tax - monthly_ni
    
    return monthly_tax, monthly_ni, net_pay

# --- TEST THE FUNCTION ---

# Calculate the gross pay using our previous variable
gross_pay = agent_data["base_salary_monthly"] + monthly_commission

# Run our new function
tax, ni, net = calculate_net_pay(gross_pay, tax_brackets_yearly)

print("--- Deductions Breakdown ---")
print(f"Gross Pay: £{gross_pay:.2f}")
print(f"Income Tax: £{tax:.2f}")
print(f"National Insurance: £{ni:.2f}")
print("-" * 28)
print(f"Net Take-Home Pay: £{net:.2f}")

--- Deductions Breakdown ---
Gross Pay: £3700.00
Income Tax: £526.67
National Insurance: £212.20
----------------------------
Net Take-Home Pay: £2961.13


In [5]:
# --- FUNCTION: GENERATE PAYSLIP ---

def generate_payslip(agent_dict, gross, comm, tax_deduct, ni_deduct, net):
    """
    Generates a formatted text-based payslip displaying all required UK components.
    """
    print("=" * 40)
    print("         OFFICIAL UK PAYSLIP")
    print("=" * 40)
    print(f"Employee Name:   {agent_dict['name']}")
    print(f"Payroll Number:  {agent_dict['agent_id']}")
    print(f"Tax Code:        S1257L") 
    print("-" * 40)
    
    print("EARNINGS")
    print(f"Base Salary:     £{agent_dict['base_salary_monthly']:>9.2f}")
    print(f"Commission:      £{comm:>9.2f}")
    print(f"Gross Pay:       £{gross:>9.2f}")
    print("-" * 40)
    
    print("DEDUCTIONS")
    print(f"Income Tax:      £{tax_deduct:>9.2f}")
    print(f"Nat. Insurance:  £{ni_deduct:>9.2f}")
    print("-" * 40)
    
    # \033[1m and \033[0m add bold text formatting in the terminal/Jupyter
    print(f"\033[1mNET TAKE-HOME PAY: £{net:>9.2f}\033[0m")
    print("=" * 40)

# --- RUN THE GENERATOR ---

# We pass in the data dictionary and all the variables we saved from our previous functions
generate_payslip(agent_data, gross_pay, monthly_commission, tax, ni, net)

         OFFICIAL UK PAYSLIP
Employee Name:   Jane Doe
Payroll Number:  AG-001
Tax Code:        S1257L
----------------------------------------
EARNINGS
Base Salary:     £  2000.00
Commission:      £  1700.00
Gross Pay:       £  3700.00
----------------------------------------
DEDUCTIONS
Income Tax:      £   526.67
Nat. Insurance:  £   212.20
----------------------------------------
NET TAKE-HOME PAY: £  2961.13


## Version 2.0: Enterprise Pandas ETL Pipeline

In [6]:
import pandas as pd

# 1. SALES ADMIN: The Master Employee Roster
df_admin = pd.DataFrame({
    'agent_id': ['AG-001', 'AG-002', 'AG-003'],
    'name': ['Jane Doe', 'John Smith', 'Alice Jones'],
    'base_salary_monthly': [2000.00, 1800.00, 2200.00]
})

# 2. UNDERWRITING: Raw Sales Data
df_underwriting = pd.DataFrame({
    'agent_id': ['AG-001', 'AG-001', 'AG-002', 'AG-003', 'AG-003'],
    'product_type': ['life_insurance', 'health_insurance', 'home_auto_insurance', 'life_insurance', 'home_auto_insurance'],
    'raw_policies_sold': [5, 15, 40, 2, 20]
})

# 3. PREMIUM ADMIN: Commission Rates & Conversions
df_premium = pd.DataFrame({
    'product_type': ['life_insurance', 'health_insurance', 'home_auto_insurance'],
    'commission_per_policy': [150.00, 50.00, 20.00],
    'conversion_rate': [0.80, 0.90, 0.95] 
})

print("Extract Phase Complete: Departmental data loaded.")

Extract Phase Complete: Departmental data loaded.


In [7]:
# Merge Underwriting and Premium data based on the 'product_type' column
# We use a 'left' merge to keep all our sales records intact, even if a product is missing from the premium list
df_sales_merged = pd.merge(df_underwriting, df_premium, on='product_type', how='left')

print("--- Merged Sales Data ---")
display(df_sales_merged)

--- Merged Sales Data ---


,agent_id,product_type,raw_policies_sold,commission_per_policy,conversion_rate
0,AG-001,life_insurance,5,150.0,0.80
1,AG-001,health_insurance,15,50.0,0.90
2,AG-002,home_auto_insurance,40,20.0,0.95
3,AG-003,life_insurance,2,150.0,0.80
4,AG-003,home_auto_insurance,20,20.0,0.95


In [8]:
# --- CALCULATE VALIDATED SALES & COMMISSION ---

# 1. Calculate validated sales
# We multiply raw sales by the conversion rate. 
# We use .round().astype(int) because you cannot have a fraction of a validated policy!
df_sales_merged['validated_sales'] = (df_sales_merged['raw_policies_sold'] * df_sales_merged['conversion_rate']).round().astype(int)

# 2. Calculate the commission for that specific product line
df_sales_merged['total_product_commission'] = df_sales_merged['validated_sales'] * df_sales_merged['commission_per_policy']

print("--- Sales & Commission Calculated ---")
display(df_sales_merged)

--- Sales & Commission Calculated ---


,agent_id,product_type,raw_policies_sold,commission_per_policy,conversion_rate,validated_sales,total_product_commission
0,AG-001,life_insurance,5,150.0,0.80,4,600.0
1,AG-001,health_insurance,15,50.0,0.90,14,700.0
2,AG-002,home_auto_insurance,40,20.0,0.95,38,760.0
3,AG-003,life_insurance,2,150.0,0.80,2,300.0
4,AG-003,home_auto_insurance,20,20.0,0.95,19,380.0


In [9]:
# --- AGGREGATE COMMISSIONS & MERGE WITH HR DATA ---

# 1. Group by agent_id and sum their total commissions
# We use .reset_index() to flatten it back into a standard, clean DataFrame
df_total_commission = df_sales_merged.groupby('agent_id')['total_product_commission'].sum().reset_index()

# Rename the column so it's clear this is their total bonus for the month
df_total_commission = df_total_commission.rename(columns={'total_product_commission': 'monthly_commission'})

# 2. Merge this total back with the master HR roster
# We use a 'left' merge on df_admin so we don't lose an employee just because they made £0 commission this month
df_payroll = pd.merge(df_admin, df_total_commission, on='agent_id', how='left')

# 3. Handle missing data (ETL Best Practice)
# If an agent sold nothing, Pandas will put 'NaN' (Not a Number). We must replace that with £0.00
df_payroll['monthly_commission'] = df_payroll['monthly_commission'].fillna(0)

# 4. Calculate Gross Pay
df_payroll['gross_pay'] = df_payroll['base_salary_monthly'] + df_payroll['monthly_commission']

print("--- Master Payroll Table ---")
display(df_payroll)

--- Master Payroll Table ---


,agent_id,name,base_salary_monthly,monthly_commission,gross_pay
0,AG-001,Jane Doe,2000.0,1300.0,3300.0
1,AG-002,John Smith,1800.0,760.0,2560.0
2,AG-003,Alice Jones,2200.0,680.0,2880.0


In [10]:
# --- APPLY TAX LOGIC TO THE ENTIRE PIPELINE ---

# We use a 'lambda' function. This is just a temporary, anonymous function that tells Pandas:
# "For every gross_pay value (x), run our calculate_net_pay function, and split the 3 results into 3 new columns."

df_payroll[['income_tax', 'national_insurance', 'net_pay']] = df_payroll['gross_pay'].apply(
    lambda x: pd.Series(calculate_net_pay(x, tax_brackets_yearly))
)

# Format the final output to look like a clean financial report
# We will round all financial columns to 2 decimal places for neatness
financial_columns = ['base_salary_monthly', 'monthly_commission', 'gross_pay', 'income_tax', 'national_insurance', 'net_pay']
df_payroll[financial_columns] = df_payroll[financial_columns].round(2)

print("=== FINAL MONTHLY PAYROLL REPORT ===")
display(df_payroll)

=== FINAL MONTHLY PAYROLL REPORT ===


,agent_id,name,base_salary_monthly,monthly_commission,gross_pay,income_tax,national_insurance,net_pay
0,AG-001,Jane Doe,2000.0,1300.0,3300.0,455.59,180.2,2664.21
1,AG-002,John Smith,1800.0,760.0,2560.0,300.19,121.0,2138.81
2,AG-003,Alice Jones,2200.0,680.0,2880.0,367.39,146.6,2366.01


In [11]:
!pip install fpdf

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40713 sha256=2cdaa76f08ac985472a81ea62a170d54431c82c38b5c1f1ef5cfbd2970daee08
  Stored in directory: c:\users\osahw\appdata\local\pip\cache\wheels\65\4f\66\bbda9866da446a72e206d6484cd97381cbc7859a7068541c36
Successfully built fpdf


In [15]:
from fpdf import FPDF

# 1. Initialize the PDF document (A4 size, measured in millimeters)
pdf = FPDF(orientation='P', unit='mm', format='A4')
pdf.add_page()

# 2. Add a Company Header (Centered)
pdf.set_font("Arial", style="B", size=16)
# w=0 means the cell stretches across the entire page width
# ln=1 tells the invisible cursor to drop down to the next line after printing
pdf.cell(w=0, h=10, txt="BRITISH INSURANCE GROUP", border=0, ln=1, align="C")

# Draw a horizontal divider line 
# (Starts 10mm from the left, 25mm from top. Ends 200mm from left, 25mm from top)
pdf.line(x1=10, y1=25, x2=200, y2=25)

# 3. Move down and add Employee Details using a standard grid
pdf.set_y(35) # Move the invisible cursor exactly 35mm from the top of the page
pdf.set_font("Arial", size=12)

# Left column: Label (40mm wide)
pdf.cell(w=40, h=10, txt="Employee Name:", border=0, align="L")
# Right column: Data (Using my details as an example. ln=1 drops to the next line)
pdf.cell(w=100, h=10, txt="Michael Edem Deblui", border=0, ln=1, align="L")

# Left column: Label
pdf.cell(w=40, h=10, txt="Payroll ID:", border=0, align="L")
# Right column: Data
pdf.cell(w=100, h=10, txt="AG-001", border=0, ln=1, align="L")

# 4. Generate the sample payslip
pdf.output("sample_payslip2.pdf")
print("PDF successfully generated!")

PDF successfully generated!


In [16]:
from fpdf import FPDF
import os

# Create a folder to store the payslips so they don't clutter your main directory
output_folder = "Payslips_Batch"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Loop through every single row in our final payroll DataFrame
for index, row in df_payroll.iterrows():
    
    # 1. Initialize the PDF for this specific agent
    pdf = FPDF(orientation='P', unit='mm', format='A4')
    pdf.add_page()
    
    # 2. Add Company Header
    pdf.set_font("Arial", style="B", size=16)
    pdf.cell(w=0, h=10, txt="BRITISH INSURANCE GROUP", border=0, ln=1, align="C")
    
    pdf.set_font("Arial", style="I", size=10)
    pdf.cell(w=0, h=8, txt="Official UK Payslip - 2026/2027 Tax Year", border=0, ln=1, align="C")
    
    # Divider Line
    pdf.line(10, 30, 200, 30)
    
    # 3. Add Agent Details
    pdf.set_y(35)
    pdf.set_font("Arial", style="B", size=12)
    
    pdf.cell(w=40, h=8, txt="Employee Name:", border=0, align="L")
    pdf.set_font("Arial", size=12)
    pdf.cell(w=100, h=8, txt=str(row['name']), border=0, ln=1, align="L")
    
    pdf.set_font("Arial", style="B", size=12)
    pdf.cell(w=40, h=8, txt="Payroll ID:", border=0, align="L")
    pdf.set_font("Arial", size=12)
    pdf.cell(w=100, h=8, txt=str(row['agent_id']), border=0, ln=1, align="L")
    
    # 4. Add Financial Breakdown
    pdf.set_y(60)
    pdf.set_font("Arial", style="B", size=12)
    pdf.cell(w=0, h=10, txt="Earnings & Deductions", border="B", ln=1, align="L")
    
    pdf.set_font("Arial", size=12)
    # Earnings
    pdf.cell(w=80, h=8, txt="Base Salary:", border=0, align="L")
    pdf.cell(w=40, h=8, txt=f"£{row['base_salary_monthly']:.2f}", border=0, ln=1, align="R")
    
    pdf.cell(w=80, h=8, txt="Commission:", border=0, align="L")
    pdf.cell(w=40, h=8, txt=f"£{row['monthly_commission']:.2f}", border=0, ln=1, align="R")
    
    pdf.cell(w=80, h=8, txt="Gross Pay:", border=0, align="L")
    pdf.cell(w=40, h=8, txt=f"£{row['gross_pay']:.2f}", border=0, ln=1, align="R")
    
    pdf.ln(5) # Add a little vertical space
    
    # Deductions
    pdf.cell(w=80, h=8, txt="Income Tax:", border=0, align="L")
    pdf.cell(w=40, h=8, txt=f"£{row['income_tax']:.2f}", border=0, ln=1, align="R")
    
    pdf.cell(w=80, h=8, txt="National Insurance:", border=0, align="L")
    pdf.cell(w=40, h=8, txt=f"£{row['national_insurance']:.2f}", border=0, ln=1, align="R")
    
    # 5. Net Pay (Boxed in)
    pdf.ln(10)
    pdf.set_font("Arial", style="B", size=14)
    pdf.cell(w=80, h=12, txt="NET TAKE-HOME PAY:", border=1, align="C")
    pdf.cell(w=40, h=12, txt=f"£{row['net_pay']:.2f}", border=1, ln=1, align="C")
    
    # 6. Save the unique PDF
    filename = f"{output_folder}/Payslip_{row['agent_id']}.pdf"
    pdf.output(filename)

print(f"Success! Generated payslips for {len(df_payroll)} agents.")

Success! Generated payslips for 3 agents.
